In [ ]:
import numpy as np 
import polars as pl 
import os
from tqdm import tqdm
import copy
import pandas as pd

import kagglehub

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix

from matplotlib import pyplot as plt
import lightgbm as lgb

from functools import partial

Thank to Chris Deotte anh his ai zoo for interesing detective @cdeotte
Thank to Naji  @najiama for his notebook.   Original dataset makes no help here. And collapse spike or millionare cliff can bring here some points at least in fold cv.

Unfortunately this solution is not orthogonal to Naji

# Make Features

In [ ]:
target = 'Will_Buy_EV'
ID_COL = "id"
#ORIG_PATH  = '/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv'

df = pl.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv')
df_test = pl.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')

#Make numerical
for c in ['Home_Charging_Possible',	'Subsidy_Available', 'Will_Buy_EV' ]:
    df=df.with_columns(pl.when(pl.col(c)=='Yes').then(1).otherwise(0).alias(c))
    try:
        df_test=df_test.with_columns(pl.when(pl.col(c)=='Yes').then(1).otherwise(0).alias(c))
    except:
        continue

df=          df.with_columns(homecharge=pl.col('Charging_Stations_Near_Home')+5* pl.col('Home_Charging_Possible'))
df_test=df_test.with_columns(homecharge=pl.col('Charging_Stations_Near_Home')+5* pl.col('Home_Charging_Possible'))


mapping = {'Low': 0, 'High': 3, 'Medium': 1}   #my own idea but correspod to Chris Deotte Fable

df = df.with_columns(pl.col("Range_Anxiety_Level").replace(mapping).cast(pl.UInt8))
df_test = df_test.with_columns(pl.col("Range_Anxiety_Level").replace(mapping).cast(pl.UInt8))

mapping = {v: i for i, v in enumerate(df["Gender"].unique().to_list())}
df = df.with_columns(pl.col("Gender").replace(mapping).cast(pl.UInt8))
df_test = df_test.with_columns(pl.col("Gender").replace(mapping).cast(pl.UInt8))

#Not working here, seems already learned by model


#df=df.with_columns(fable=1.2*pl.col('Annual_Income_USD' )/100000 + 0.6*pl.col('Environmental_Concern_Level')+2*pl.col('Subsidy_Available')-pl.col('Range_Anxiety_Level')-5.5)
#df=df.with_columns(worry_scorry = pl.col('Daily_Commute_km') - 5*pl.col('Charging_Stations_Near_Home') - 5 *pl.col('Charging_Stations_Near_Work') - 150 * pl.col('Home_Charging_Possible') )


# Couple functions

In [ ]:

def focal_loss_fobj_general(preds, train_data, gamma = 1.2, alpha = 0.65 ):
    labels = train_data.get_label()          # <-- вот так достаём y_true

    p = 1.0 / (1.0 + np.exp(-preds))
    eps = 1e-7
    p = np.clip(p, eps, 1 - eps)
    pt = np.where(labels == 1, p, 1 - p)
    focal_weight = (1 - pt) ** gamma
    alpha_factor = np.where(labels == 1, alpha, 1 - alpha)*2
    weight = alpha_factor * focal_weight

    grad = weight * (p - labels)
    hess = weight * p * (1 - p)
    hess = np.clip(hess, 1e-7, None)

    return grad, hess

def smoothed_target_encoding(X_train, y_train, col, kernel_param=35.0):
    """
    Target encoding with gauss smoothing
    """

    grouped = y_train.groupby(X_train[col]).agg(['mean', 'count']).reset_index()
    cats = grouped[col].values.astype(float)   # unique cats values
    means = grouped['mean'].values.astype(float)
    counts = grouped['count'].values.astype(float)

    # gauss kernel
    diff = cats[:, None] - cats[None, :]
    K = np.exp(-kernel_param * diff * diff)

    # meaning
    numerator = np.sum(K * (means * counts)[None, :], axis=1)
    denominator = np.sum(K * counts[None, :], axis=1)
    denominator = np.maximum(denominator, 1e-12)   # защита от 0
    smoothed = numerator / denominator

    return dict(zip(cats, smoothed))

# LGBM

In [ ]:

focal_loss_fobj = partial(focal_loss_fobj_general, alpha=0.65, gamma=1.2) #fix alpha and gamma here

tecols=[ 'Age', 'Gender', 'Number_of_Cars_Owned',  'City_Type', 'Current_Car_Type',  'Range_Anxiety_Level', ] # For TE encoding
tegausscols=['Annual_Income_USD' ]  # For gasuss kernel te encoding

dropcols=[ 'Age', 'Gender','Number_of_Cars_Owned', 'City_Type', 'Current_Car_Type', 'Charging_Stations_Near_Home'] #not for model

features = [col for col in df.columns if col not in ['id', target]] 
cat_cols= [] 



X_full = df.to_pandas()[features]
y_full = df.to_pandas()[target]#.values
X_test = df_test.to_pandas()[features]


params = {
    'objective': focal_loss_fobj , #'binary',
    'metric':  'auc',
    'boosting_type':  'gbdt',
    'num_leaves': 63, 
    'learning_rate': 0.02,
    'feature_fraction': 0.3, 
    'bagging_fraction': 0.9,
    'bagging_freq': 1, 
    'verbose': -1,
    'random_state': 42,
    'reg_alpha': 0.05,
    'reg_lambda': 10.0,
    'max_bin' : 1023,
    'n_jobs':-1,
    #'device_type': 'gpu',
    'min_data_in_leaf': 10,  
   # 'scale_pos_weight': 3.0,  #torn on if binary 
    'linear_tree': True
}


# ----------------------------
# 4. Cross validation on oof
# ----------------------------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

test_probas = []  # 
alphate=0.5
#  importances lists
gain_importances = []
split_importances = []
weights=[]

oof_preds = np.zeros([len(X_full)])

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
        
    print(f"\n=== Fold {fold+1} ===")
    
    # Split to val
    X_train, y_train = X_full.iloc[train_idx], y_full[train_idx] 
    X_val,   y_val= X_full.iloc[val_idx],   y_full[val_idx]  

    X_train_te = X_train.copy()
    X_val_te = X_val.copy()
    X_test_te = X_test.copy()
    
    # --- Target Encoding  ---################################
    global_mean = y_train.mean()
    
    for col in tecols:
        # train stat
        counts = y_train.groupby(X_train[col]).size()
        means = y_train.groupby(X_train[col]).mean()
        smoothed = (counts * means + alphate * global_mean) / (counts + alphate)
        te_col = f'te_{col}'
        
        X_train_te[te_col] = X_train[col].map(smoothed).fillna(global_mean)
        X_val_te[te_col] = X_val[col].map(smoothed).fillna(global_mean)
        X_test_te[te_col] = X_test_te[col].map(smoothed).fillna(global_mean)

    # --- Target Encoding with gauss ---#####################
    
    ker_l=[0.02] #different features may require different parameter!
    for k, col in enumerate(tegausscols):
        smoothed = smoothed_target_encoding(X_train, y_train, col, kernel_param=ker_l[k])

        te_col = f'te_{col}'
        X_train_te[te_col] = X_train[col].map(smoothed).fillna(global_mean)
        X_val_te[te_col] = X_val[col].map(smoothed).fillna(global_mean)
        X_test_te[te_col] = X_test_te[col].map(smoothed).fillna(global_mean)
        

     #--- Frequency Encoding  --- ###########################
    for col in tecols:
        freq = X_train[col].value_counts()

        freq_col = f'{col}_freq'
        X_train_te[freq_col] = X_train[col].map(freq).fillna(0)
        X_val_te[freq_col] = X_val[col].map(freq).fillna(0)
        X_test_te[freq_col] = X_test_te[col].map(freq).fillna(0)

    
    X_train_te=X_train_te.drop(dropcols, axis=1)
    X_val_te=X_val_te.drop(dropcols, axis=1)
    X_test_te=X_test_te.drop(dropcols, axis=1)

    for c in cat_cols:
        X_train_te[c] = X_train_te[c].fillna('unknown').astype('category')
        X_val_te[c]   = X_val_te[c].fillna('unknown').astype('category')
        X_test_te[c]   = X_test_te[c].fillna('unknown').astype('category')

    
    if fold==0: 
        print(X_train_te.columns)
        display(X_train_te.describe())

    train_data = lgb.Dataset(X_train_te, label=y_train , categorical_feature=cat_cols)#, init_score=(Xi_train.values)) #logit
    val_data   = lgb.Dataset(X_val_te,   label=y_val,   reference=train_data , categorical_feature=cat_cols)#, init_score=(Xi_val.values)) #logit
    
    
    model = lgb.train( params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    num_boost_round=5000,
    callbacks=[lgb.early_stopping(150), lgb.log_evaluation(50)]    )

    # -------------------------------------------------
    # Val ROC AUC 
    # -------------------------------------------------
    val_pred = model.predict(X_val_te, num_iteration=model.best_iteration)#, raw_score=True) + (Xi_val.values)

    # keep  OOF-п
    oof_preds[val_idx] = val_pred
    
    val_auc = roc_auc_score(y_val, val_pred)
    print(f"  Validation ROC AUC = {val_auc:.4f}")
    weights.append(val_auc)

    cm = confusion_matrix(y_val, val_pred>0.5)
    print("Confusion matrix:")
    print(cm)

    # Feature Importances
    gain_importances.append(model.feature_importance(importance_type='gain'))
    split_importances.append(model.feature_importance(importance_type='split'))
    
    pred_test = model.predict(X_test_te, num_iteration=model.best_iteration)#, raw_score=True)+ (Xi_test.values)
    test_probas.append(pred_test)


# mean for submission:
final_pred = np.mean(test_probas, axis=0)
print(roc_auc_score(y_full, oof_preds))

# Feature importances

In [ ]:
ff=X_train_te.columns

# mean
avg_gain = np.mean(gain_importances, axis=0)
avg_split = np.mean(split_importances, axis=0)

importance_df = pd.DataFrame({
    'feature': ff,
    'avg_gain': avg_gain,
    'avg_split': avg_split
}).sort_values('avg_gain', ascending=False)

print(importance_df)

In [ ]:
pl.read_csv('/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv').with_columns(Will_Buy_EV=final_pred).write_csv('submission.csv')